# AutismAssist AI — ASD Screening Model

Build a baseline classification pipeline with preprocessing, Random Forest, and evaluation metrics.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score

df = pd.read_csv("../data/raw/asd_screening_dataset.csv")
target = "ASD_Status"
features = [c for c in df.columns if c not in [target, "Child_ID"]]
cat = ["Gender"]
num = [c for c in features if c not in cat]

X_train, X_test, y_train, y_test = train_test_split(
    df[features], df[target], test_size=0.2, random_state=42, stratify=df[target]
)


In [ ]:
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
])

model = Pipeline([
    ("preprocessor", pre),
    ("model", RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced"
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, pred))
print("Accuracy:", accuracy_score(y_test, pred))
print("F1:", f1_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))


In [ ]:
# Save the model for the Streamlit app
import joblib
joblib.dump(model, "../models/asd_screening_model.joblib")


## Interpretation

The model output is a screening-support signal. It should not be presented as a diagnosis. Model quality must be externally validated on representative clinical data before any real-world use.